<a href="https://colab.research.google.com/github/ms914/3DKWM_server/blob/main/MWP/KWM_CLI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Kugelwolkenmodell (MWP)

Dieses Notebook zeigt in genialer Weise, wie man innerhalb von colab eine Frontend-Backend Struktur aufbaut. In einem weiteren Schritt wird die Logik und Datenstruktur im Backend erweitert. Die Logik implementiert eine Bindungslogik mit Ausrichtung der Atome mit CGA-Rotor. Der CGA-Rotor enthält die Rotation und die Translation. Das CLI bittet den Controller, eine automatische Annäherung durchzuführen. Vorteil: klare Trennung von commandline interface und mathematisch-chemischer Logik.
Das CLI interface ist rein textbasiert. Die Bindungslogik gibt den Wolkenstatus aus.

// 1 = reaktiv (•), 2 = frei (:), 3 = gebunden (≡)

Das entspricht der Lewis Schreibweise.

## CLI Interface Client

# Pseudocode

IMPORT app_controller

FUNKTION starte_cli_interface():
    controller = app_controller.Initialisiere()
    print("=== KUGELWOLKEN-MODELL CLI V1.0 ===")
    
    // Hauptschleife (Event-Loop des Terminals)
    SOLANGE_LAUFT:
        zeiche_aktuelle_welt(controller.hole_welt_zustand())
        print("\nBefehle: [erzeuge <Symbol>] | [binde <AtomA> <AtomB>] | [exit]")
        eingabe = input("Eingabe> ")
        
        WENN eingabe == "exit":
            BRECHE_AB
            
        SONST WENN eingabe.STARTET_MIT("erzeuge"):
            symbol = eingabe.SPLIT()[1] // z.B. "C" oder "H"
            controller.handle_erzeuge_atom(symbol)
            print(f"-> Atom {symbol} generiert.")
            
        SONST WENN eingabe.STARTET_MIT("binde"):
            teile = eingabe.SPLIT()
            atom_A_id = teile[1]
            atom_B_id = teile[2]
            
            // Das CLI bittet den Controller, eine automatische Annäherung
            // der jeweils nächsten freien Wolken zu simulieren
            erfolg = controller.handle_binde_naechste_wolken(atom_A_id, atom_B_id)
            
            WENN erfolg:
                print("-> Bindung erfolgreich geometrisch berechnet!")
            SONST:
                print("-> Fehler: Keine freien Wolken oder Atome nicht gefunden.")
                
        SONST:
            print("-> Unbekannter Befehl.")

FUNKTION zeiche_aktuelle_welt(welt_zustand):
    print("\n" + "-"*50)
    print("AKTUELLE ATOME IN DER SZENE:")
    print("-"*50)
    print("UID      | SYMBOL | POSITION (X, Y, Z)     | WOLKEN (Status)")
    print("-"*50)
    
    FUER JEDES uid, atom IN welt_zustand:
        pos_str = f"[{atom.position[0]:.2f}, {atom.position[1]:.2f}, {atom.position[2]:.2f}]"
        
        // Wolken-Status kompakt als String darstellen
        // 1 = reaktiv (•), 2 = frei (:), 3 = gebunden (≡)
        wolken_status = ""
        FUER wolke IN atom.wolken:
            WENN wolke.elektronen == 1: wolken_status += " •"
            WENN wolke.elektronen == 2: wolken_status += " :"
            WENN wolke.elektronen == 3: wolken_status += " ≡"
            
        print(f"{uid:<8} | {atom.symbol:<6} | {pos_str:<22} | [{wolken_status.strip()}]")
    print("-"*50)

backend

In [6]:
# 1. Server-Code in Datei schreiben
code = """
from fastapi import FastAPI
import uvicorn

app = FastAPI()

WELT_ZUSTAND = {}

@app.get("/status")
def get_status():
    return WELT_ZUSTAND

@app.post("/erzeuge")
def erzeuge(daten: dict):
    symbol = daten["symbol"]
    uid = f"{symbol}_1"
    WELT_ZUSTAND[uid] = {"symbol": symbol, "position": [0,0,0]}
    return {"status": "success", "uid": uid}
"""
with open("server.py", "w") as f:
    f.write(code)

# 2. Server im Hintergrund starten und Logs unterdrücken
import subprocess
import time
subprocess.Popen(["uvicorn", "server:app", "--port", "8000"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(2) # Kurz warten, bis der Server hochgefahren ist
print("⚡ FastAPI-Server läuft im Hintergrund auf Port 8000!")

⚡ FastAPI-Server läuft im Hintergrund auf Port 8000!


frontend

In [7]:
import requests
import json

class ColabCliController:
    def __init__(self, base_url="http://127.0.0.1:8000"):
        self.base_url = base_url

    def hole_welt_zustand(self):
        try:
            return requests.get(f"{self.base_url}/status").json()
        except:
            return {}

    def handle_erzeuge_atom(self, symbol):
        requests.post(f"{self.base_url}/erzeuge", json={"symbol": symbol})

def zeige_welt(welt_zustand):
    print("\n" + "="*40)
    print("AKTUELLE WELT (VOM SERVER):")
    print("="*40)
    if not welt_zustand:
        print(" Keine Atome vorhanden.")
    for uid, daten in welt_zustand.items():
        print(f"ID: {uid} | Element: {daten['symbol']} | Pos: {daten['position']}")
    print("="*40)

controller (startet frontend)

In [8]:
controller = ColabCliController()

while True:
    zustand = controller.hole_welt_zustand()
    zeige_welt(zustand)

    eingabe = input("\nBefehl [erzeuge <Symbol> / exit]: ").strip()

    if eingabe == "exit":
        print("CLI beendet.")
        break
    elif eingabe.startswith("erzeuge "):
        symbol = eingabe.split(" ")[1]
        controller.handle_erzeuge_atom(symbol)
    else:
        print("Unbekannter Befehl.")


AKTUELLE WELT (VOM SERVER):
ID: C_1 | Element: C | Pos: [-0.2654652724523654, 0.2654075093023138, 0.2758570456638604]
ID: C_2 | Element: C | Pos: [0.4562225640346669, 0.9870953457893461, 0.9975448821508927]
ID: C_3 | Element: C | Pos: [-0.0717487412448371, 0.26973091296646734, 0.041776412180101086]
ID: H_4 | Element: H | Pos: [0.6499390952421952, 0.9914187494534996, 0.7634642486671334]
ID: H_5 | Element: H | Pos: [-0.4449422011026728, 0.3433334477584946, -0.03437182696576657]

Befehl [erzeuge <Symbol> / exit]: exit
CLI beendet.


Resultat: Hier läuft ein Server Endpoint in colab. Er wird durch ein CLI-Frontend angesteuert. Per Texteingabe verändert man den Weltstatus im Backend.

## CGA Endpoint

Dieses MWP ist genial: Es reduziert die wesentlichen Punkte des Kugelwolkenmodells und implementiert die Bindungslogik in CGA. Der Rotor Bivektor ist das Ergebnis.

Das kann man auslesen und an three.js übergeben und als mesh darstellen lassen.

kanns nicht glauben ...

In [1]:
!pip install clifford

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.0/160.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.9/151.9 kB 9.5 MB/s eta 0:00:00


Daten: Chemie

Logik: CGA

In [2]:
code = """
from fastapi import FastAPI
import numpy as np
from clifford.g3c import * # Importiert vordefinierten CGA-Raum (Conformal G3)
import uvicorn

app = FastAPI()
WELT_ZUSTAND = {}

# --- CGA HILFSFUNKTIONEN ---
def cga_punkt(x, y, z):
    \"\"\"Erzeugt einen konformen Punkt aus 3D-Koordinaten.\"\"\"
    return up(x*e1 + y*e2 + z*e3)

def cga_extrahiere_3d(punkt_cga):
    \"\"\"Extrahiert die klassischen 3D-Koordinaten aus einem CGA-Punkt.\"\"\"
    # down() projiziert den Punkt zurück in den e1, e2, e3 Raum
    p_3d = down(punkt_cga)
    return [float(p_3d[e1]), float(p_3d[e2]), float(p_3d[e3])]

def berechne_richtungs_rotor(v_start, v_ziel):
    \"\"\"Erzeugt einen CGA-Rotor, der v_start in v_ziel überführt (R = sqrt(v_ziel * v_start)).\"\"\"
    v_s = (v_start[0]*e1 + v_start[1]*e2 + v_start[2]*e3).normal()
    v_z = (v_ziel[0]*e1 + v_ziel[1]*e2 + v_ziel[2]*e3).normal()

    # Geometrisches Produkt
    gp = v_z * v_s
    # Skalarer Anteil + Bivektor-Anteil für die Wurzel-Approximation bei Rotoren
    return (1 + gp).normal()

# --- REINES DATEN-MODELL ---
ELEMENT_PROFIL = {
    "H":  {"geo": "KUGEL",     "valenzelektronen": 1, "radius": 0.18, "abstand": 1.25},
    "C":  {"geo": "TETRAEDER", "valenzelektronen": 4, "radius": 0.30, "abstand": 1.25},
}

def hole_lokale_wolken(symbol):
    if symbol == "H":
        return [[0.0, 0.0, 0.0]]
    elif symbol == "C":
        # Vereinfachte Tetraeder-Vektoren
        return [[1, 1, 1], [-1, -1, 1], [-1, 1, -1], [1, -1, -1]]
    return [[0.0, 0.0, 0.0]]

# --- LOGIK ---
@app.get("/status")
def get_status():
    return WELT_ZUSTAND

@app.post("/erzeuge")
def erzeuge(daten: dict):
    symbol = daten["symbol"]
    uid = f"{symbol}_{len(WELT_ZUSTAND) + 1}"

    # Start-Zustand: Einheits-Rotor (keine Drehung)
    WELT_ZUSTAND[uid] = {
        "symbol": symbol,
        "position": (np.random.rand(3) - 0.5).tolist(),
        "rotor_bivektor": [0.0, 0.0, 0.0], # Repräsentiert die Drehung im CLI
        "wolken": [{"id": i, "elektronen": 1, "lokal": v} for i, v in enumerate(hole_lokale_wolken(symbol))]
    }
    return {"status": "success", "uid": uid}

@app.post("/binde")
def binde(daten: dict):
    atom_A = WELT_ZUSTAND[daten["atom_A_id"]]
    atom_B = WELT_ZUSTAND[daten["atom_B_id"]]

    # Finde erste freie Wolken im Backend
    w_A = next(w for w in atom_A["wolken"] if w["elektronen"] < 3)
    w_B = next(w for w in atom_B["wolken"] if w["elektronen"] < 3)

    # 1. Geometrische Achse bestimmen basierend auf Wolke A (Translations-Ziel)
    achse = np.array(w_A["lokal"])
    if np.linalg.norm(achse) == 0:  # Falls H (Kugel/Ursprung), nutze Abstandsvektor
        achse = np.array([1.0, 0.0, 0.0])
    else:
        achse = achse / np.linalg.norm(achse)

    fester_abstand = ELEMENT_PROFIL[atom_A["symbol"]]["abstand"]

    # 2. Position von B exakt fixieren (Translation via CGA-Punkt-Generierung)
    neue_pos_B = np.array(atom_A["position"]) + (achse * fester_abstand)
    atom_B["position"] = neue_pos_B.tolist()

    # 3. Orientierung für B berechnen (Rotor Gesicht-zu-Gesicht)
    # Zielrichtung für B ist das genaue Inverse der Wolkenrichtung von A
    ziel_richtung_B = -achse
    start_richtung_B = np.array(w_B["lokal"])
    if np.linalg.norm(start_richtung_B) == 0:
        start_richtung_B = np.array([1.0, 0.0, 0.0])

    # CGA-Rotor berechnen
    R = berechne_richtungs_rotor(start_richtung_B, ziel_richtung_B)

    # Speichere die Orientierung (hier vereinfacht als Bivektor-Muster für das JSON)
    atom_B["rotor_bivektor"] = [float(R[e1*e2]), float(R[e2*e3]), float(R[e3*e1])]

    # Status auf 'gebunden' setzen
    w_A["elektronen"] = 3
    w_B["elektronen"] = 3

    return {"status": "success"}
"""

with open("server.py", "w") as f:
    f.write(code)

import subprocess
import time
subprocess.Popen(["uvicorn", "server:app", "--port", "8000"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(2)
print("⚡ CGA-FastAPI-Server läuft im Hintergrund auf Port 8000!")

⚡ CGA-FastAPI-Server läuft im Hintergrund auf Port 8000!


In [4]:
import requests

class CgaCliController:
    def __init__(self, base_url="http://127.0.0.1:8000"):
        self.base_url = base_url

    def hole_zustand(self):
        return requests.get(f"{self.base_url}/status").json()

    def erzeuge(self, symbol):
        return requests.post(f"{self.base_url}/erzeuge", json={"symbol": symbol}).json()

    def binde(self, id_a, id_b):
        return requests.post(f"{self.base_url}/binde", json={"atom_A_id": id_a, "atom_B_id": id_b}).json()

def zeige_welt(welt):
    print("\n" + "="*70)
    print("CGA-MOLEKÜL-STATUS:")
    print("="*70)
    for uid, data in welt.items():
        pos = [f"{x:.2f}" for x in data['position']]
        rot = [f"{x:.2f}" for x in data['rotor_bivektor']]
        wolken = "".join([" ≡" if w['elektronen']==3 else " •" for w in data['wolken']])
        print(f"ID: {uid:<6} | Pos: {str(pos):<22} | Rotor-Bivek: {str(rot):<22} | Wolken:[{wolken} ]")
    print("="*70)

# CLI-Loop
cc = CgaCliController()
while True:
    zeige_welt(cc.hole_zustand())
    eingabe = input("Befehl [erzeuge <C/H> | binde <id1> <id2> | exit]: ").strip().split()

    if not eingabe or eingabe[0] == "exit":
        break
    elif eingabe[0] == "erzeuge" and len(eingabe) > 1:
        cc.erzeuge(eingabe[1].upper())
    elif eingabe[0] == "binde" and len(eingabe) > 2:
        res = cc.binde(eingabe[1], eingabe[2])
        if "error" in res: print("Fehler:", res["error"])


CGA-MOLEKÜL-STATUS:
ID: C_1    | Pos: ['-0.27', '0.27', '0.28'] | Rotor-Bivek: ['0.00', '0.00', '0.00'] | Wolken:[ ≡ • • • ]
ID: C_2    | Pos: ['0.46', '0.99', '1.00'] | Rotor-Bivek: ['0.00', '0.00', '0.00'] | Wolken:[ ≡ • • • ]
ID: C_3    | Pos: ['-0.07', '0.27', '0.04'] | Rotor-Bivek: ['0.00', '0.00', '0.00'] | Wolken:[ ≡ • • • ]
ID: H_4    | Pos: ['0.65', '0.99', '0.76'] | Rotor-Bivek: ['0.63', '0.00', '0.63'] | Wolken:[ ≡ ]
ID: H_5    | Pos: ['-0.44', '0.34', '-0.03'] | Rotor-Bivek: ['0.00', '0.00', '0.00'] | Wolken:[ • ]
Befehl [erzeuge <C/H> | binde <id1> <id2> | exit]: exit


next: CH4, H2O, O=O, Smiles-Eingabe, Lewis-Ausgabe